In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd", "Catalog")
dbutils.widgets.text("bronze_schema", "natalkamartinuk55_bronze", "Bronze schema")
dbutils.widgets.text("raw_volume_path", "/Volumes/dbr_dev_ua5816bd/natalkamartinuk55/raw_data", "Raw volume path")
dbutils.widgets.text("sample_video_id", "-0CMnp02rNY", "Sample video_id")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_volume_path = dbutils.widgets.get("raw_volume_path")
sample_video_id = dbutils.widgets.get("sample_video_id")

BRONZE_TABLE = f"{catalog}.{bronze_schema}.youtube_videos_files_bronze"

In [0]:
from pyspark.sql.functions import col

df_existing_row = (
    spark.table(BRONZE_TABLE)
    .filter(col("video_id") == sample_video_id)
    .orderBy(col("trending_date").desc())
    .limit(1)
)

display(df_existing_row.select("video_id", "views", "trending_date"))

In [0]:
import time
from pyspark.sql.functions import date_format

batch_id = int(time.time())
new_batch_path = f"{raw_volume_path}/simulated_new_batch_{batch_id}"

df_new_batch = (
    df_existing_row
    .withColumn("views", (col("views").cast("long") + 15000).cast("string"))
    .withColumn("trending_date", date_format(current_date(), "yy.dd.MM"))
    .drop("_source_file", "_ingested_at", "_rescued_data")
)

(
    df_new_batch
    .coalesce(1)
    .write
    .format("csv")
    .option("header", "true")
    .save(new_batch_path)
)

In [0]:
display(dbutils.fs.ls(new_batch_path))

In [0]:
%sql
SELECT video_id, views, trending_date, __START_AT, __END_AT
FROM dbr_dev_ua5816bd.natalkamartinuk55_silver.youtube_videos_ldp_scd2
WHERE video_id = '-0CMnp02rNY'
ORDER BY __START_AT;